# Introduction to Time-Series Forecasting
### Teaching Notebook | Data Science Education Series
**Author:** Md Minhazur Rahman  
**MSc Data Science (Merit)** — University of Greenwich  
**Preprint:** [DOI: 10.5281/zenodo.19479285](https://doi.org/10.5281/zenodo.19479285)  
**Live Research Demo:** [Streamlit Dashboard](https://retail-forecasting-hvdzvesi4u9l6fs5tvdoyi.streamlit.app/)  

---

> **Who is this for?**  
> This notebook is designed for undergraduate students with basic Python knowledge who want to understand time-series data and build their first forecasting model from scratch.

---

## Section 1: What is Time-Series Data?

A **time-series** is simply a sequence of data points recorded at regular time intervals.

You encounter time-series data every day without realising it:

| Example | What is measured | How often |
|---|---|---|
| Daily shop sales | Revenue in BDT | Every day |
| Temperature readings | Degrees Celsius | Every hour |
| Stock prices | Share price | Every minute |
| Website traffic | Number of visitors | Every day |

### The key feature that makes time-series special:
> **The order of the data points matters.**  
> Monday's sales affect Tuesday's stock level.  
> Yesterday's temperature affects today's forecast.  
> You cannot shuffle the rows of a time-series the way you might shuffle a standard dataset.

### Real Example: Daily Sales at a Chittagong Retail Shop

Imagine a small electronics shop in Chittagong. Every day, the owner records how many units of a product were sold. Over 30 days, this gives us a time-series of 30 data points.

Let us visualise what this looks like — first using a simple synthetic (artificially generated) pattern.

In [ ]:
# Import the libraries we need
import numpy as np
import matplotlib.pyplot as plt

# Set a random seed so results are reproducible
np.random.seed(42)

# Generate 90 days of time points
days = np.arange(1, 91)

# Create a synthetic sales pattern:
# - A sine wave captures the weekly rhythm of sales
# - We add random noise to make it realistic
sales = 50 + 15 * np.sin(2 * np.pi * days / 7) + np.random.normal(0, 5, size=len(days))

# Plot the result
plt.figure(figsize=(12, 4))
plt.plot(days, sales, color='steelblue', linewidth=1.5, label='Daily Sales (Units)')
plt.axhline(y=50, color='red', linestyle='--', alpha=0.5, label='Average Sales (50 units)')
plt.title('Synthetic Daily Sales Data — 90 Days', fontsize=14, fontweight='bold')
plt.xlabel('Day')
plt.ylabel('Units Sold')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Average daily sales: {sales.mean():.1f} units')
print(f'Maximum daily sales: {sales.max():.1f} units')
print(f'Minimum daily sales: {sales.min():.1f} units')

**What do you notice in this plot?**

- Sales go up and down in a regular weekly pattern — this is called **seasonality**
- There is random variation around the pattern — this is called **noise**
- The average stays around 50 units — this is called the **level** or **baseline**

These three components — **level, seasonality, and noise** — appear in almost every real-world time-series.

---

## Section 2: Why Does Forecasting Matter?

**Forecasting** means using past data to predict future values.

### The Business Problem: Stock Replenishment

Return to our Chittagong electronics shop. The owner faces a real problem every week:

> *"How many units should I order from my supplier today, so that I do not run out of stock — but also do not over-order and waste money?"*

This is a **demand forecasting** problem. If the owner can predict next week's sales with reasonable accuracy:

| Without forecasting | With forecasting |
|---|---|
| Orders based on gut feeling | Orders based on data |
| Frequent stockouts or overstock | Optimised inventory levels |
| Lost sales or wasted capital | Reduced costs, higher service level |
| Reactive decisions | Proactive planning |

### The Scale of This Problem in Bangladesh

Bangladesh's retail and e-commerce sector is growing rapidly. Platforms like Chaldal, Shajgoj, and Pathao Food all face this exact problem at massive scale — thousands of SKUs, millions of daily transactions. Accurate forecasting is not a luxury; it is a competitive necessity.

This is precisely why my MSc dissertation focused on building a **privacy-preserving forecasting pipeline** — one that works even when real customer data cannot be shared due to privacy regulations.

---

## Section 3: Your First Forecasting Model

We will now build a simple but complete forecasting pipeline in four steps:

1. Generate synthetic sales data
2. Create a time-based feature
3. Split into training and test sets
4. Fit a Linear Regression model and evaluate it

### Why Linear Regression as a starting point?

Linear Regression is the simplest forecasting model. It assumes sales can be predicted by a straight-line relationship with time. It will not be our best model — but it gives us a **baseline** to beat with more advanced methods later.

> **Important concept: The Train-Test Split**  
> In time-series, we always train on the **past** and test on the **future**.  
> We never shuffle the data randomly — that would be cheating, because in real life you cannot use tomorrow's data to predict today.

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ── Step 1: Generate 100 days of synthetic sales data ──────────────────
np.random.seed(42)
n_days = 100
day_index = np.arange(1, n_days + 1)

# Sales = upward trend + weekly seasonality + random noise
trend     = 0.3 * day_index
seasonal  = 10 * np.sin(2 * np.pi * day_index / 7)
noise     = np.random.normal(0, 4, size=n_days)
sales     = 40 + trend + seasonal + noise

# Build a clean DataFrame
df = pd.DataFrame({
    'day':   day_index,
    'sales': sales
})

print('Dataset shape:', df.shape)
print(df.head(10).to_string(index=False))

In [ ]:
# ── Step 2: Train-Test Split (80 days train / 20 days test) ────────────
# We split by TIME — not randomly
split_point = 80

train = df[df['day'] <= split_point]
test  = df[df['day'] >  split_point]

print(f'Training set: days 1 to {split_point}  ({len(train)} rows)')
print(f'Test set:     days {split_point+1} to {n_days} ({len(test)} rows)')

# Define features and target
X_train = train[['day']]
y_train = train['sales']

X_test  = test[['day']]
y_test  = test['sales']

In [ ]:
# ── Step 3: Fit Linear Regression ─────────────────────────────────────
model = LinearRegression()
model.fit(X_train, y_train)

# Generate predictions on the test set
y_pred = model.predict(X_test)

# ── Step 4: Evaluate the model ─────────────────────────────────────────
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print('── Model Performance on Test Set ──')
print(f'MAE  (Mean Absolute Error):       {mae:.2f} units')
print(f'RMSE (Root Mean Squared Error):   {rmse:.2f} units')
print(f'MAPE (Mean Absolute % Error):     {mape:.2f}%')
print()
print('Interpretation:')
print(f'  On average, our model is off by {mae:.1f} units per day.')
print(f'  This is a {mape:.1f}% error relative to actual sales.')

In [ ]:
# ── Step 5: Visualise actual vs predicted ──────────────────────────────
plt.figure(figsize=(13, 5))

# Plot training data
plt.plot(train['day'], train['sales'],
         color='steelblue', linewidth=1.5,
         label='Training Data (Days 1–80)')

# Plot actual test data
plt.plot(test['day'], test['sales'],
         color='darkorange', linewidth=1.5,
         label='Actual Sales (Test Period)')

# Plot model predictions
plt.plot(test['day'], y_pred,
         color='green', linewidth=2,
         linestyle='--', label='Linear Regression Forecast')

# Add a vertical line at the train-test boundary
plt.axvline(x=split_point, color='red',
            linestyle=':', alpha=0.7, label='Train / Test Split')

plt.title('Linear Regression Forecast vs Actual Sales', fontsize=14, fontweight='bold')
plt.xlabel('Day')
plt.ylabel('Units Sold')
plt.legend()
plt.tight_layout()
plt.show()

### Reading the Results

Look at the plot carefully:

- The **green dashed line** is what our model predicted
- The **orange line** is what actually happened
- The gap between them is our **forecast error**

**Why is Linear Regression not perfect here?**

Linear Regression can only learn a **straight-line trend**. It cannot capture the weekly ups and downs (seasonality). This is why our MAPE is not zero — the model misses the pattern.

This is the fundamental lesson:
> *A simple model gives us a baseline. Our job as data scientists is to beat that baseline with smarter models.*

---

## Section 4: What Comes Next?

You have just built your first forecasting model. Here is the roadmap for going further:

### 3 Advanced Topics to Explore

**1. Random Forest Regression**
- Instead of one straight line, Random Forest builds hundreds of decision trees and averages their predictions
- It can capture non-linear patterns that Linear Regression misses
- Typically reduces MAE by 20–40% on retail data

**2. XGBoost (Extreme Gradient Boosting)**
- One of the most powerful algorithms for tabular time-series data
- Builds trees sequentially, each one correcting the errors of the last
- Used in production by major e-commerce and retail companies worldwide

**3. Rolling-Origin Cross-Validation (ROCV)**
- A smarter way to evaluate time-series models
- Instead of one train-test split, we simulate multiple forecasting scenarios
- Gives a more reliable estimate of real-world model performance
- This is the validation method used in my MSc dissertation

### Explore the Full Research

This notebook is the foundation. My MSc dissertation extends these ideas into a complete, production-grade pipeline:

- **Full codebase and documentation:**  
  [github.com/minhazda/synthetic-retail-forecasting](https://github.com/minhazda/synthetic-retail-forecasting)

- **Live interactive dashboard — try it yourself:**  
  [retail-forecasting-hvdzvesi4u9l6fs5tvdoyi.streamlit.app](https://retail-forecasting-hvdzvesi4u9l6fs5tvdoyi.streamlit.app/)

- **Published preprint (DOI-indexed):**  
  [doi.org/10.5281/zenodo.19479285](https://doi.org/10.5281/zenodo.19479285)

---

### Discussion Questions for Students

1. What would happen to our forecast if sales suddenly doubled during Eid? How would Linear Regression handle this?
2. Why is it dangerous to use future data when training a forecasting model?
3. Can you modify the code above to add a lag feature — yesterday's sales as a predictor of today's sales?
4. What are the ethical implications of using synthetic data instead of real customer data for model training?

---

*These materials are developed for undergraduate data science education in Bangladesh.*  
*© Md Minhazur Rahman, 2026. Open for educational use.*